# Vorlagen und Hilfsfunktionen

Dieses Notebook enthält Codevorlagen, die du für dein Bildfilter-Projekt verwenden kannst:

1. **Vorlage 1**: Filter, die jeden Pixel einzeln umrechnen (Pixel-für-Pixel).
2. **Vorlage 2**: Filter, die zuerst ein leeres Bild erstellen und dann Pixel an bestimmten Positionen einsetzen.
3. **Hilfsfunktion**: Ein Video laden, jeden Frame filtern und das Ergebnis als neues Video speichern.

Kopier dir die Vorlagen, die du brauchst, in dein eigenes Projekt-Notebook und passe sie an.

---

## Vorbereitung: Bild laden und speichern

Wie in **Schritt 0** des Skripts. Führe diese Zelle einmal aus, damit `lade_bild` und `speichere_bild` zur Verfügung stehen.

In [ ]:
from PIL import Image

def lade_bild(pfad):
    img = Image.open(pfad).convert("RGB")
    breite, hoehe = img.size
    bild = []
    for y in range(hoehe):
        zeile = []
        for x in range(breite):
            zeile.append(img.getpixel((x, y)))
        bild.append(zeile)
    return bild

def speichere_bild(bild, pfad):
    hoehe = len(bild)
    breite = len(bild[0])
    img = Image.new("RGB", (breite, hoehe))
    for y in range(hoehe):
        for x in range(breite):
            img.putpixel((x, y), bild[y][x])
    img.save(pfad)

---

## Vorlage 1: Pixel-für-Pixel

Eignet sich für Filter, bei denen jeder neue Pixel nur vom gleichen Pixel im Originalbild abhängt, zum Beispiel:

- Invertieren
- Graustufen
- Helligkeit
- Schwellwert
- Einen Farbkanal ausschalten

Ändere in der Vorlage nur die `TODO`-Zeile.

In [ ]:
def mein_filter(bild):
    neues_bild = []
    for y, zeile in enumerate(bild):
        neue_zeile = []
        for x, pixel in enumerate(zeile):
            r, g, b = pixel
            # TODO: Berechne hier den neuen Pixel aus (r, g, b)
            neue_zeile.append((r, g, b))
        neues_bild.append(neue_zeile)
    return neues_bild

### Beispiel: Rot ausschalten

Hier ist die Vorlage 1 für den Filter `kein_rot` ausgefüllt. Der Rot-Wert wird auf 0 gesetzt.

In [ ]:
def kein_rot(bild):
    neues_bild = []
    for y, zeile in enumerate(bild):
        neue_zeile = []
        for x, pixel in enumerate(zeile):
            r, g, b = pixel
            neue_zeile.append((0, g, b))
        neues_bild.append(neue_zeile)
    return neues_bild

# Anwenden:
# bild = lade_bild("dein_foto.jpg")
# speichere_bild(kein_rot(bild), "ohne_rot.png")

---

## Vorlage 2: Zuerst leeres Bild, dann Pixel platzieren

Eignet sich für Filter, bei denen der neue Pixel an einer anderen Position liegt als der ursprüngliche, oder bei denen ganze Bereiche bearbeitet werden, zum Beispiel:

- Horizontal/vertikal spiegeln
- Drehen
- Verpixeln
- Bild verschieben oder kacheln

Du erstellst zuerst ein leeres `neues_bild`. In der zweiten Schleife liest du `bild[y][x]` und schreibst das Ergebnis an eine frei wählbare Position in `neues_bild`.

In [ ]:
def mein_filter(bild):
    hoehe = len(bild)
    breite = len(bild[0])

    # 1. Leeres neues Bild erstellen (mit schwarzen Pixeln gefuellt)
    neues_bild = []
    for y in range(hoehe):
        zeile = []
        for x in range(breite):
            zeile.append((0, 0, 0))
        neues_bild.append(zeile)

    # 2. Pixel platzieren - du darfst an beliebige Stellen schreiben!
    for y in range(hoehe):
        for x in range(breite):
            r, g, b = bild[y][x]
            neuer_pixel = (0, g, b)
            neues_bild[y][x] = neuer_pixel

    return neues_bild

---

## Hilfsfunktion: Video filtern

`bearbeite_video` lädt ein Video, wendet einen deiner Filter auf jeden einzelnen Frame an und setzt das Video wieder zusammen.

**Achtung:** Pixel-für-Pixel-Filter in reinem Python sind langsam. Ein 5-Sekunden-Clip mit 720p hat schon ~7 Millionen Pixel pro Frame × 150 Frames. Teste zuerst mit einem kurzen Video in niedriger Auflösung.

In [ ]:
import imageio
import numpy as np

def bearbeite_video(eingabe_pfad, ausgabe_pfad, filter_funktion):
    reader = imageio.get_reader(eingabe_pfad)
    fps = reader.get_meta_data().get("fps", 24)
    writer = imageio.get_writer(ausgabe_pfad, fps=fps)

    for i, frame in enumerate(reader):
        hoehe, breite = frame.shape[:2]
        bild = []
        for y in range(hoehe):
            zeile = []
            for x in range(breite):
                r, g, b = frame[y, x]
                zeile.append((int(r), int(g), int(b)))
            bild.append(zeile)

        gefiltert = filter_funktion(bild)
        out_frame = np.array(gefiltert, dtype=np.uint8)
        writer.append_data(out_frame)
        print(f"Frame {i + 1} bearbeitet")

    reader.close()
    writer.close()
    print(f"Fertig: {ausgabe_pfad}")

### Beispiel: `kein_rot` auf ein Video anwenden

Stelle sicher, dass du die Zellen für `kein_rot` und `bearbeite_video` weiter oben ausgeführt hast.

In [ ]:
# bearbeite_video("katze.mp4", "katze_ohne_rot.mp4", kein_rot)